# Chapter 1 — Reliable, Scalable, and Maintainable Applications

STAR
Situataion:
1. hard to combine tools when you need to do something that a single tool cannot do alone.
2. 

Task:
1. explore
what different tools have in common, what distinguishes them, and how they achieve their characteristics.

Action:

reliable, scalable, and maintainable data systems.

Result:

data-intensive application.

```mermaid
flowchart TB

    ROOT["Data-Intensive<br/>Applications"]

    ROOT --> REL["RELIABILITY"]
    ROOT --> SCA["SCALABILITY"]
    ROOT --> MAI["MAINTAINABILITY"]

    REL --> R1["Tolerating hardware<br/>&amp; software faults"]
    REL --> R2["Human error"]

    SCA --> S1["Measuring load<br/>&amp; performance"]
    S1 --> S2["Latency percentiles"]
    S1 --> S3["Throughput"]

    MAI --> M1["Operability"]
    MAI --> M2["Simplicity<br/>&amp; evolvability"]

    classDef root fill:#faf7f0,stroke:#222,stroke-width:3px,color:#222,font-size:20px;
    classDef group fill:#f5f0e6,stroke:#555,stroke-width:2px,color:#222,font-size:18px;
    classDef item fill:#fffdf8,stroke:#999,stroke-width:1px,color:#333,font-size:14px;

    class ROOT root;
    class REL,SCA,MAI group;
    class R1,R2,S1,S2,S3,M1,M2 item;
```

## Reliability

<div style="border-left: 5px solid #E76F51; background: #FDEDE7; color: #2B2B2B; padding: 10px 14px; margin: 8px 0; border-radius: 4px;">
<strong style="color: #C1502E;">Tolerating hardware &amp; software faults</strong> — the system keeps working correctly even when things go wrong. Hardware faults (disk crashes, faulty RAM, power outages) are usually random and independent, and are handled with redundancy. Software faults are systematic bugs — harder to anticipate because they're correlated across nodes and can cascade (e.g. a bug triggered by a specific input, runaway resource use, or one service's failure taking down another).
</div>

<div style="border-left: 5px solid #F4A261; background: #FEF4E9; color: #2B2B2B; padding: 10px 14px; margin: 8px 0; border-radius: 4px;">
<strong style="color: #B5651D;">Human error</strong> — operators are the least reliable part of the system (config mistakes are a leading cause of outages). Reduce impact by minimizing opportunities for error, decoupling places where people make mistakes from places that cause failures (sandboxes with real data, but no real consequences), testing thoroughly at every level, allowing fast and easy recovery, and setting up detailed monitoring.
</div>

### Most famous example — Netflix Chaos Monkey

Netflix pioneered deliberately breaking its own production systems to prove they can tolerate it. **Chaos Monkey** randomly terminates live VM instances in production; the wider "Simian Army" goes further, simulating a whole availability-zone outage, network latency spikes, and even config/human-error scenarios — all while real traffic keeps flowing.

```mermaid
flowchart TB
    C["Client"] --> ELB["Elastic Load Balancer"]
    ELB --> AZ1
    ELB --> AZ2
    ELB --> AZ3

    subgraph AZ1["Availability Zone A"]
        I1["EC2 instance"]
    end

    subgraph AZ2["Availability Zone B"]
        I2["EC2 instance"]
        CM["Chaos Monkey<br/>terminates instance"]
        CM -.kills.-> I2
        ASG["Auto Scaling Group<br/>replaces instance"]
        ASG -.spawns.-> I2
    end

    subgraph AZ3["Availability Zone C"]
        I3["EC2 instance"]
    end

    I1 --> DB[("Cassandra<br/>multi-region replicated")]
    I2 --> DB
    I3 --> DB

    classDef az fill:#FDEDE7,stroke:#E76F51,stroke-width:2px,color:#2B2B2B;
    classDef node fill:#fffdf8,stroke:#999,stroke-width:1px,color:#2B2B2B;
    classDef chaos fill:#F4A261,stroke:#B5651D,stroke-width:2px,color:#2B2B2B;
    class AZ1,AZ2,AZ3 az;
    class I1,I2,I3,ASG,DB node;
    class CM chaos;
```

**Where each Reliability point is implemented:**

<div style="border-left: 5px solid #E76F51; background: #FDEDE7; color: #2B2B2B; padding: 10px 14px; margin: 8px 0; border-radius: 4px;">
<strong style="color: #C1502E;">Tolerating hardware &amp; software faults</strong> → the <em>Auto Scaling Group</em> detects the killed instance and spawns a replacement, the <em>Elastic Load Balancer</em> routes traffic away from AZ B while it recovers, and <em>Cassandra's</em> multi-region replication means no data is lost even if an entire AZ's nodes disappear.
</div>

<div style="border-left: 5px solid #F4A261; background: #FEF4E9; color: #2B2B2B; padding: 10px 14px; margin: 8px 0; border-radius: 4px;">
<strong style="color: #B5651D;">Human error</strong> → Chaos Monkey turns "an instance disappearing" into a routine, rehearsed event instead of a surprise a human operator would otherwise have to react to at 3am — it's the deliberate, controlled version of the mistake someone will eventually make for real.
</div>

## Scalability

<div style="border-left: 5px solid #2A9D8F; background: #E9F6F4; color: #2B2B2B; padding: 10px 14px; margin: 8px 0; border-radius: 4px;">
<strong style="color: #1F7A6E;">Measuring load &amp; performance</strong> — load is described with <em>load parameters</em> (e.g. requests per second, ratio of reads to writes, number of concurrently active users). Performance is then examined by asking either "if load increases and resources stay fixed, how does performance suffer?" or "how much do resources need to grow to keep performance unchanged?"
</div>

<div style="border-left: 5px solid #264653; background: #E8ECEE; color: #2B2B2B; padding: 10px 14px; margin: 8px 0; border-radius: 4px;">
<strong style="color: #264653;">Latency percentiles</strong> — response time should be treated as a <em>distribution</em> of values, not a single number. Percentiles (median/p50, p95, p99, p999) reveal what a typical vs. a worst-case user experiences; high percentiles ("tail latencies") matter because they're often felt by the customers with the most data or highest-value requests.
</div>

<div style="border-left: 5px solid #6A4C93; background: #F1ECF6; color: #2B2B2B; padding: 10px 14px; margin: 8px 0; border-radius: 4px;">
<strong style="color: #5B3D82;">Throughput</strong> — the number of records or requests the system can process per unit time; typically the metric that matters most for batch-processing systems, as opposed to response time for online systems.
</div>

### Most famous example — Twitter's home-timeline fan-out

The case study Kleppmann himself uses in the book. Posting a tweet is cheap, but rendering a user's home timeline means merging tweets from everyone they follow. At Twitter's scale — a read:write ratio in the hundreds-to-one, and celebrities with tens of millions of followers — computing that join on every read doesn't scale, so the system needs a deliberate architectural choice about *where* the fan-out work happens.

```mermaid
flowchart TB
    subgraph WRITE["Fan-out on write — used for most users"]
        direction LR
        U1["User posts tweet"] --> FO["Fan-out service"]
        FO --> TL1["Follower A<br/>cached timeline"]
        FO --> TL2["Follower B<br/>cached timeline"]
        FO --> TL3["Follower C<br/>cached timeline"]
    end

    subgraph READ["Fan-out on read — fallback for celebrities"]
        direction LR
        U2["Celebrity posts tweet"] --> TS[("Tweet store")]
        R1["Follower requests<br/>home timeline"] --> M["Merge at read time"]
        TS --> M
    end

    classDef write fill:#E9F6F4,stroke:#2A9D8F,stroke-width:2px,color:#2B2B2B;
    classDef read fill:#F1ECF6,stroke:#6A4C93,stroke-width:2px,color:#2B2B2B;
    class U1,FO,TL1,TL2,TL3 write;
    class U2,TS,R1,M read;
```

**Where each Scalability point is implemented:**

<div style="border-left: 5px solid #2A9D8F; background: #E9F6F4; color: #2B2B2B; padding: 10px 14px; margin: 8px 0; border-radius: 4px;">
<strong style="color: #1F7A6E;">Measuring load &amp; performance</strong> → Twitter picked between the two designs by measuring its actual load parameters (tweets/sec vs. timeline reads/sec) and asking how each design's performance held up as that ratio grew.
</div>

<div style="border-left: 5px solid #6A4C93; background: #F1ECF6; color: #2B2B2B; padding: 10px 14px; margin: 8px 0; border-radius: 4px;">
<strong style="color: #5B3D82;">Throughput</strong> → <em>fan-out on write</em> trades write throughput (one tweet becomes millions of timeline-cache writes) for cheap, high-throughput reads (a home-timeline read is just a cache lookup).
</div>

<div style="border-left: 5px solid #264653; background: #E8ECEE; color: #2B2B2B; padding: 10px 14px; margin: 8px 0; border-radius: 4px;">
<strong style="color: #264653;">Latency percentiles</strong> → fanning a celebrity's tweet out to tens of millions of followers synchronously would spike p99 write latency for everyone; falling back to <em>fan-out on read</em> for high-follower accounts keeps the tail latency bounded across the whole system.
</div>

## Maintainability

<div style="border-left: 5px solid #588157; background: #EFF5EC; color: #2B2B2B; padding: 10px 14px; margin: 8px 0; border-radius: 4px;">
<strong style="color: #40663F;">Operability</strong> — make it easy for operations teams to keep the system running smoothly: good visibility into system health, straightforward ways to manage the system, and predictable behavior that avoids surprises.
</div>

<div style="border-left: 5px solid #C9A227; background: #FCF6E7; color: #2B2B2B; padding: 10px 14px; margin: 8px 0; border-radius: 4px;">
<strong style="color: #96781C;">Simplicity &amp; evolvability</strong> — <em>simplicity</em> means managing complexity through good abstractions so new engineers can understand the system; <em>evolvability</em> (agility) means making it easy to modify the system safely as requirements change over time.
</div>

### Most famous example — Etsy's continuous deployment

Etsy popularized deploying to production dozens of times a day, paired with **blameless postmortems** (a practice championed by John Allspaw) — treating operability and safe evolvability as first-class engineering concerns instead of an afterthought bolted on after the "real" work is done.

```mermaid
flowchart LR
    Dev["Engineer<br/>commits code"] --> CI["CI: build + test"]
    CI --> Flag["Deploy behind<br/>feature flag (off)"]
    Flag --> Prod["Production"]
    Prod --> Dash["Dashboards &amp; alerts<br/>(e.g. StatsD/Graphite)"]
    Dash --> Ramp["Gradually ramp<br/>feature flag %"]
    Ramp --> Flag
    Dash -->|"incident"| PM["Blameless<br/>postmortem"]
    PM --> Dev

    classDef op fill:#EFF5EC,stroke:#588157,stroke-width:2px,color:#2B2B2B;
    classDef evo fill:#FCF6E7,stroke:#C9A227,stroke-width:2px,color:#2B2B2B;
    class Dash,PM op;
    class Dev,CI,Flag,Prod,Ramp evo;
```

**Where each Maintainability point is implemented:**

<div style="border-left: 5px solid #588157; background: #EFF5EC; color: #2B2B2B; padding: 10px 14px; margin: 8px 0; border-radius: 4px;">
<strong style="color: #40663F;">Operability</strong> → <em>dashboards and alerts</em> give operators real-time visibility into system health, and the <em>blameless postmortem</em> loop turns every incident into a system improvement instead of assigning blame.
</div>

<div style="border-left: 5px solid #C9A227; background: #FCF6E7; color: #2B2B2B; padding: 10px 14px; margin: 8px 0; border-radius: 4px;">
<strong style="color: #96781C;">Simplicity &amp; evolvability</strong> → the <em>feature flag</em> decouples deployment from release, so a change can ship to production continuously and be ramped up, down, or rolled back independently — evolving the system safely without a big-bang release.
</div>